# 📖 Actividad 2: Lectura de los Datasets con Pandas
---
**Objetivo:** Cargar las 3 fuentes de datos principales (MIDAGRI, INDECI, AGRARIA.PE) en DataFrames de Pandas. Documentar las fuentes originales y generar archivos intermedios para las siguientes actividades.

### Fuentes Documentadas
| Fuente | URL de Referencia | Formato |
|:-------|:------------------|:--------|
| MIDAGRI | [https://app.powerbi.com/view?r=...SISAGRI](https://www.gob.pe/midagri) | Excel (.xlsx) |
| INDECI | [https://www.gob.pe/indeci](https://www.gob.pe/indeci) — SINPAD / Datos Abiertos | Excel (.xlsx) + Shapefile (.dbf) |
| Agraria.pe | [https://www.agraria.pe/](https://www.agraria.pe/) | CSV (scraping) |
| NASA POWER | [https://power.larc.nasa.gov/](https://power.larc.nasa.gov/) | CSV (API NASA) |


In [1]:
# ==========================================================================
# 2.0 Carga de Configuración del Pipeline
# ==========================================================================
import os, sys, json, glob, warnings
import unicodedata
import pandas as pd
import numpy as np
from dbfread import DBF

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
# Encoding (no aplica en Jupyter kernel)
# if sys.platform == 'win32':
#     sys.stdout.reconfigure(encoding='utf-8')

# Navegar a la raíz del proyecto (necesario si se ejecuta desde notebooks/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(PROJECT_ROOT)
print(f"Directorio de trabajo: {os.getcwd()}")

# Cargar configuración de la Actividad 1
with open(os.path.join('data', '02_interim', 'pipeline_config.json'), 'r', encoding='utf-8') as f:
    CONFIG = json.load(f)

DIRS = CONFIG['DIRS']
ANHO_INICIO = CONFIG['ANHO_INICIO']
ANHO_FIN = CONFIG['ANHO_FIN']
CULTIVO_TARGET = CONFIG['CULTIVO_TARGET']

print("✅ Configuración cargada desde Actividad 1.")
print(f"   Rango temporal: {ANHO_INICIO} - {ANHO_FIN}")
print(f"   Cultivo target: {CULTIVO_TARGET}")


Directorio de trabajo: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-
✅ Configuración cargada desde Actividad 1.
   Rango temporal: 2021 - 2025
   Cultivo target: LIMON


## 2.1 Lectura de MIDAGRI (SISAGRI)
Fuente oficial del Ministerio de Desarrollo Agrario y Riego.  
Contiene producción agrícola por cultivo, departamento, provincia y distrito a nivel mensual.


In [2]:
# ==========================================================================
# 2.1 MIDAGRI — Sistema Integrado de Estadísticas Agrarias (SISAGRI)
# ==========================================================================
print("=" * 70)
print("  LECTURA FUENTE 1: MIDAGRI (SISAGRI)")
print("=" * 70)

midagri_path = os.path.join(DIRS['raw_midagri'], 'Sisagri_2016_2025.xlsx')

# Lectura de la hoja 2021-2025
df_midagri_full = pd.read_excel(midagri_path, sheet_name='2021_2025', engine='openpyxl')

print(f"\n  📊 Hoja leída: '2021_2025'")
print(f"  Filas totales: {len(df_midagri_full):,}")
print(f"  Columnas: {df_midagri_full.columns.tolist()}")
print(f"  Tipos de dato:")
print(df_midagri_full.dtypes.to_string())

# Filtro por cultivo LIMON
cultivo_col = df_midagri_full['dsc_Cultivo'].astype(str).str.upper().str.strip()
mask_limon = cultivo_col.str.contains(CULTIVO_TARGET, case=False, na=False)

# Filtro temporal
df_midagri_full['anho'] = pd.to_numeric(df_midagri_full['anho'], errors='coerce')
mask_tiempo = df_midagri_full['anho'].between(ANHO_INICIO, ANHO_FIN)

df_midagri = df_midagri_full.loc[mask_limon & mask_tiempo].copy()

print(f"\n  🍋 Registros de LIMÓN ({ANHO_INICIO}-{ANHO_FIN}): {len(df_midagri):,}")
print(f"  Variantes encontradas: {df_midagri['dsc_Cultivo'].unique().tolist()}")
print(f"  Departamentos con limón: {df_midagri['Dpto'].nunique()}")

# Vista previa
print("\n  Muestra de datos:")
print(df_midagri[['anho', 'mes', 'Dpto', 'Prov', 'dsc_Cultivo', 'PRODUCCION(t)', 'MTO_PRECCHAC (S/ x kg)']].head(5).to_string(index=False))


  LECTURA FUENTE 1: MIDAGRI (SISAGRI)



  📊 Hoja leída: '2021_2025'
  Filas totales: 778,065
  Columnas: ['anho', 'mes', 'COD_UBIGEO', 'Dpto', 'Prov', 'Dist', 'dsc_Cultivo', 'PRODUCCION(t)', 'COSECHA (ha)', 'MTO_PRECCHAC (S/ x kg)']
  Tipos de dato:
anho                        int64
mes                         int64
COD_UBIGEO                  int64
Dpto                          str
Prov                          str
Dist                          str
dsc_Cultivo                   str
PRODUCCION(t)             float64
COSECHA (ha)              float64
MTO_PRECCHAC (S/ x kg)    float64



  🍋 Registros de LIMÓN (2021-2025): 19,573
  Variantes encontradas: ['LIMON', 'LIMON DULCE']
  Departamentos con limón: 23

  Muestra de datos:
 anho  mes     Dpto        Prov dsc_Cultivo  PRODUCCION(t)  MTO_PRECCHAC (S/ x kg)
 2021    1 AMAZONAS CHACHAPOYAS       LIMON          18.21                     1.5
 2021    1 AMAZONAS CHACHAPOYAS       LIMON           4.95                     1.5
 2021    1 AMAZONAS       BAGUA       LIMON          15.00                     1.5
 2021    1 AMAZONAS       BAGUA       LIMON           2.20                     2.5
 2021    1 AMAZONAS       BAGUA LIMON DULCE           6.00                     1.3


## 2.2 Lectura de INDECI (SINPAD)
Se cargan **dos tipos de fuentes**:
1. **Resúmenes consolidados** (2003-2024): Tablas agregadas por departamento y provincia. Útiles para el EDA geográfico.
2. **Registros evento-a-evento** (DBFs 2021-2023): Data granular con fecha, tipo de fenómeno y afectados. Estos alimentan la serie temporal del LSTM.


In [3]:
# ==========================================================================
# 2.2a INDECI — Resúmenes Consolidados (Excel)
# ==========================================================================
print("\n" + "=" * 70)
print("  LECTURA FUENTE 2a: INDECI — Resúmenes Consolidados")
print("=" * 70)

# --- Resumen por Departamento ---
em_path = os.path.join(DIRS['raw_indeci'], 'resumen_emergencias_2003_2024.xlsx')
df_indeci_dpto = pd.read_excel(em_path, sheet_name='POR DPTO', header=None)

# El header real está en las filas 4-6 (merged cells)
# Fila 6 tiene los sub-headers: AFECT, DAMNIF, DESAP, LESION, FALLEC, etc.
col_names = [
    'departamento', 'emergencias',
    'pers_afectadas', 'pers_damnificadas', 'pers_desaparecidas',
    'pers_lesionadas', 'pers_fallecidas',
    'viv_afectadas', 'viv_destruidas',
    'salud_afectadas', 'salud_destruidas',
    'cultivo_has_afectadas', 'cultivo_has_perdidas',
    'puentes_afectados', 'puentes_perdidos',
    'carreteras_km_afectadas', 'carreteras_km_perdidas'
]

# Saltar las 7 primeras filas (encabezado complejo)
df_indeci_dpto = pd.read_excel(em_path, sheet_name='POR DPTO', header=None, skiprows=7)
df_indeci_dpto.columns = col_names[:len(df_indeci_dpto.columns)]

# Eliminar la fila TOTAL y filas vacías
df_indeci_dpto = df_indeci_dpto.dropna(subset=['departamento'])
df_indeci_dpto = df_indeci_dpto[df_indeci_dpto['departamento'].astype(str).str.strip() != 'TOTAL']
df_indeci_dpto = df_indeci_dpto[~df_indeci_dpto['departamento'].astype(str).str.contains('Fuente|NOTA', na=True)]

print(f"  Departamentos en resumen INDECI: {len(df_indeci_dpto)}")
print(f"  Columnas: {df_indeci_dpto.columns.tolist()}")

# --- Resumen por Departamento + Provincia ---
df_indeci_prov = pd.read_excel(em_path, sheet_name='POR DPTO_PROV', header=None, skiprows=7)
df_indeci_prov.columns = col_names[:len(df_indeci_prov.columns)]
df_indeci_prov = df_indeci_prov.dropna(subset=['departamento'])
df_indeci_prov = df_indeci_prov[df_indeci_prov['departamento'].astype(str).str.strip() != 'TOTAL']

print(f"  Filas en resumen por provincia: {len(df_indeci_prov)}")



  LECTURA FUENTE 2a: INDECI — Resúmenes Consolidados
  Departamentos en resumen INDECI: 27
  Columnas: ['departamento', 'emergencias', 'pers_afectadas', 'pers_damnificadas', 'pers_desaparecidas', 'pers_lesionadas', 'pers_fallecidas', 'viv_afectadas', 'viv_destruidas', 'salud_afectadas', 'salud_destruidas', 'cultivo_has_afectadas', 'cultivo_has_perdidas', 'puentes_afectados', 'puentes_perdidos', 'carreteras_km_afectadas', 'carreteras_km_perdidas']
  Filas en resumen por provincia: 224


In [4]:
# ==========================================================================
# 2.2b INDECI — Registros Evento a Evento (DBFs del SINPAD)
# ==========================================================================
print("\n" + "=" * 70)
print("  LECTURA FUENTE 2b: INDECI — DBFs Evento a Evento (2021-2023)")
print("=" * 70)

dbf_files = {
    2021: os.path.join(DIRS['raw_indeci'], 'E_2021', 'Emergencias_2021.dbf'),
    2022: os.path.join(DIRS['raw_indeci'], 'E_2022', 'Emergencias_2022.dbf'),
    2023: os.path.join(DIRS['raw_indeci'], 'E_2023', 'E_2023.dbf'),
}

dfs_dbf = []
for year, dbf_path in dbf_files.items():
    if os.path.exists(dbf_path):
        table = DBF(dbf_path, encoding='latin1', load=True)
        df_year = pd.DataFrame(list(table))
        df_year.columns = [str(c).lower() for c in df_year.columns]
        dfs_dbf.append(df_year)
        print(f"  ✅ {year}: {len(df_year):,} registros | Columnas: {len(df_year.columns)}")
    else:
        print(f"  ⚠️  {year}: Archivo no encontrado en {dbf_path}")

if dfs_dbf:
    df_indeci_eventos = pd.concat(dfs_dbf, ignore_index=True)
    print(f"\n  Total eventos combinados: {len(df_indeci_eventos):,}")
    print(f"  Campos clave: {['ide_sinpad', 'fecha', 'departamen', 'provincia', 'fenomeno', 'safecta', 'sdamni', 'sareaculti', 'sareacul_1']}")
    print(f"  Muestra:")
    print(df_indeci_eventos[['ide_sinpad', 'fecha', 'departamen', 'provincia', 'fenomeno']].head(3).to_string(index=False))
else:
    df_indeci_eventos = pd.DataFrame()
    print("  ⚠️  No se cargaron datos de eventos INDECI.")



  LECTURA FUENTE 2b: INDECI — DBFs Evento a Evento (2021-2023)


  ✅ 2021: 11,175 registros | Columnas: 77


  ✅ 2022: 11,574 registros | Columnas: 79


  ✅ 2023: 2,276 registros | Columnas: 74

  Total eventos combinados: 25,025
  Campos clave: ['ide_sinpad', 'fecha', 'departamen', 'provincia', 'fenomeno', 'safecta', 'sdamni', 'sareaculti', 'sareacul_1']
  Muestra:
 ide_sinpad      fecha departamen   provincia         fenomeno
   132749.0 12/01/2021   AMAZONAS CHACHAPOYAS LLUVIAS INTENSAS
   135727.0 02/03/2021   AMAZONAS CHACHAPOYAS LLUVIAS INTENSAS
   137608.0 30/03/2021   AMAZONAS CHACHAPOYAS LLUVIAS INTENSAS


## 2.3 Lectura de Noticias Agrícolas (Agraria.pe)
Noticias web scrapeadas del portal Agraria.pe, cubriendo el período 2021-2025.  
Contienen: fecha, titular, cuerpo completo, fuente (sección) y URL.


In [5]:
# ==========================================================================
# 2.3 AGRARIA.PE — Noticias Agrícolas
# ==========================================================================
print("\n" + "=" * 70)
print("  LECTURA FUENTE 3: AGRARIA.PE — Noticias Agrícolas")
print("=" * 70)

news_files = sorted(glob.glob(os.path.join(DIRS['raw_news'], 'agro_news_*.csv')))
print(f"  Archivos encontrados: {len(news_files)}")

dfs_news = []
for f in news_files:
    df_n = pd.read_csv(f)
    year = os.path.basename(f).replace('agro_news_', '').replace('.csv', '')
    print(f"  📰 {os.path.basename(f)}: {len(df_n)} noticias")
    dfs_news.append(df_n)

df_noticias = pd.concat(dfs_news, ignore_index=True)
df_noticias['fecha'] = pd.to_datetime(df_noticias['fecha'], errors='coerce')
df_noticias = df_noticias.dropna(subset=['fecha']).sort_values('fecha')

print(f"\n  Total noticias consolidadas: {len(df_noticias)}")
print(f"  Rango temporal: {df_noticias['fecha'].min()} → {df_noticias['fecha'].max()}")
print(f"  Columnas: {df_noticias.columns.tolist()}")
print(f"  Categorías (fuente):")
print(df_noticias['fuente'].value_counts().to_string())



  LECTURA FUENTE 3: AGRARIA.PE — Noticias Agrícolas
  Archivos encontrados: 5
  📰 agro_news_2021.csv: 44 noticias
  📰 agro_news_2022.csv: 133 noticias
  📰 agro_news_2023.csv: 118 noticias
  📰 agro_news_2024.csv: 108 noticias
  📰 agro_news_2025.csv: 125 noticias

  Total noticias consolidadas: 528
  Rango temporal: 2021-01-04 00:00:00 → 2025-12-22 00:00:00
  Columnas: ['fecha', 'titular', 'cuerpo_completo', 'fuente', 'url']
  Categorías (fuente):
fuente
agraria.pe/produccion                199
agraria.pe/politica                  148
agraria.pe/negocios                   66
agraria.pe/clima-y-medio-ambiente     35
agraria.pe/proyectos                  24
agraria.pe/eventos                    17
agraria.pe/salud-y-sanidad            15
agraria.pe/alimentacion               14
agraria.pe/tecnologia                 10


In [6]:
# ==========================================================================
# 2.4 NASA POWER — Datos Climáticos
# ==========================================================================
print("\n" + "=" * 70)
print("  LECTURA FUENTE 4: NASA POWER — Datos Climáticos")
print("=" * 70)

# El compañero de tesis procesó los datos en el pipeline especializado.
# Leemos los datos intermedios unificados para validar su estructura.
nasa_path = os.path.join(DIRS['interim_nasa'], 'nasa_long_raw.csv')
if os.path.exists(nasa_path):
    df_nasa = pd.read_csv(nasa_path)
    print(f"  ✅ NASA POWER cargado: {len(df_nasa):,} registros")
    print(f"  Columnas: {df_nasa.columns.tolist()}")
    print(f"  Rango temporal: {df_nasa['ANIO'].min()}-{df_nasa['MES'].min()} → {df_nasa['ANIO'].max()}-{df_nasa['MES'].max()}")
    print(f"  Variables: T2M, T2M_MAX, T2M_MIN, PRECTOTCORR, RH2M, WS2M, ALLSKY_SFC_SW_DWN")
    display(df_nasa.head(3))
else:
    print("  ⚠️  NASA POWER: Archivo nasa_long_raw.csv no encontrado.")
    print("      Asegúrate de ejecutar el pipeline NASA primero.")



  LECTURA FUENTE 4: NASA POWER — Datos Climáticos
  ✅ NASA POWER cargado: 6,120 registros
  Columnas: ['DEPARTAMENTO', 'PROVINCIA', 'ANIO', 'MES', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M']
  Rango temporal: 2021-1 → 2025-12
  Variables: T2M, T2M_MAX, T2M_MIN, PRECTOTCORR, RH2M, WS2M, ALLSKY_SFC_SW_DWN


,DEPARTAMENTO,PROVINCIA,ANIO,MES,ALLSKY_SFC_SW_DWN,PRECTOTCORR,QV2M,RH2M,T2M,T2M_MAX,T2M_MIN,WS2M
0,AMAZONAS,BAGUA,2021,1,14.94,1.42,13.15,72.32,21.86,30.12,15.28,1.85
1,AMAZONAS,BAGUA,2021,2,14.92,1.24,13.00,67.00,23.07,34.49,16.68,1.70
2,AMAZONAS,BAGUA,2021,3,14.31,3.40,13.53,77.16,21.12,30.10,15.44,1.49


## 2.4 Guardado de Datos Intermedios
Se guardan los DataFrames cargados en `data/02_interim/` para ser consumidos por las actividades siguientes sin necesidad de releer los archivos originales pesados.


In [7]:
# ==========================================================================
# 2.4 Exportación a Archivos Intermedios
# ==========================================================================
print("\n" + "=" * 70)
print("  GUARDADO DE DATOS INTERMEDIOS")
print("=" * 70)

interim_dir = DIRS['interim']

# MIDAGRI
midagri_interim = os.path.join(interim_dir, 'midagri_limon_raw.csv')
df_midagri.to_csv(midagri_interim, index=False, encoding='utf-8-sig')
print(f"  ✅ MIDAGRI → {midagri_interim} ({len(df_midagri):,} filas)")

# INDECI Resumen Dpto
indeci_dpto_interim = os.path.join(interim_dir, 'indeci_resumen_dpto.csv')
df_indeci_dpto.to_csv(indeci_dpto_interim, index=False, encoding='utf-8-sig')
print(f"  ✅ INDECI Resumen Dpto → {indeci_dpto_interim} ({len(df_indeci_dpto)} filas)")

# INDECI Resumen Prov
indeci_prov_interim = os.path.join(interim_dir, 'indeci_resumen_prov.csv')
df_indeci_prov.to_csv(indeci_prov_interim, index=False, encoding='utf-8-sig')
print(f"  ✅ INDECI Resumen Prov → {indeci_prov_interim} ({len(df_indeci_prov)} filas)")

# INDECI Eventos (DBFs)
if not df_indeci_eventos.empty:
    indeci_eventos_interim = os.path.join(interim_dir, 'indeci_eventos_dbf.csv')
    df_indeci_eventos.to_csv(indeci_eventos_interim, index=False, encoding='utf-8-sig')
    print(f"  ✅ INDECI Eventos → {indeci_eventos_interim} ({len(df_indeci_eventos):,} filas)")

# Noticias
noticias_interim = os.path.join(interim_dir, 'agraria_noticias_raw.csv')
df_noticias.to_csv(noticias_interim, index=False, encoding='utf-8-sig')
print(f"  ✅ Noticias → {noticias_interim} ({len(df_noticias)} filas)")

print(f"  Archivos generados: midagri_limon_raw.csv, indeci_resumen_dpto.csv,")
print(f"    indeci_resumen_prov.csv, indeci_eventos_dbf.csv, agraria_noticias_raw.csv")

# Integración NASA (Copiado de interim_nasa si existe)
nasa_interim = os.path.join(DIRS['interim_nasa'], 'nasa_long_raw.csv')
if os.path.exists(nasa_interim):
    print(f"  ✅ NASA Datos Listos para Fase 3 (EDA).")

print(f"\n[ACTIVIDAD 2] COMPLETADA.")
print(f"  Descripción: Lectura de MIDAGRI, INDECI y AGRARIA.PE en DataFrames.")
print(f"  Resultado: 5 archivos intermedios generados en {interim_dir}")
print(f"  Archivos generados: midagri_limon_raw.csv, indeci_resumen_dpto.csv,")
print(f"    indeci_resumen_prov.csv, indeci_eventos_dbf.csv, agraria_noticias_raw.csv")



  GUARDADO DE DATOS INTERMEDIOS
  ✅ MIDAGRI → data\02_interim\midagri_limon_raw.csv (19,573 filas)
  ✅ INDECI Resumen Dpto → data\02_interim\indeci_resumen_dpto.csv (27 filas)
  ✅ INDECI Resumen Prov → data\02_interim\indeci_resumen_prov.csv (224 filas)


  ✅ INDECI Eventos → data\02_interim\indeci_eventos_dbf.csv (25,025 filas)
  ✅ Noticias → data\02_interim\agraria_noticias_raw.csv (528 filas)
  Archivos generados: midagri_limon_raw.csv, indeci_resumen_dpto.csv,
    indeci_resumen_prov.csv, indeci_eventos_dbf.csv, agraria_noticias_raw.csv
  ✅ NASA Datos Listos para Fase 3 (EDA).

[ACTIVIDAD 2] COMPLETADA.
  Descripción: Lectura de MIDAGRI, INDECI y AGRARIA.PE en DataFrames.
  Resultado: 5 archivos intermedios generados en data\02_interim
  Archivos generados: midagri_limon_raw.csv, indeci_resumen_dpto.csv,
    indeci_resumen_prov.csv, indeci_eventos_dbf.csv, agraria_noticias_raw.csv
